# Block-Diagonal Inference for Large Graphs

When a knowledge graph has hundreds of propositions, a single factor graph
won't fit on TSU hardware — each p-bit has only ~12 connections.
**Block-diagonal decomposition** solves this:

1. **Partition** the graph into small blocks (2–4 propositions)
2. **Sample** — run exact Gibbs sampling within each block
3. **Message-pass** — send marginal messages between blocks (BP-style)
4. **Iterate** until convergence

This notebook walks through the full pipeline on a 5-node chain.

In [ ]:
%pip install -e ..

In [ ]:
from pln_thrml import (
    partition_into_blocks,
    run_block_diagonal_sampling,
    sample_and_measure_block_diagonal,
    build_beta_full_graph,
    run_beta_sampling, estimate_beta_marginal,
)

---
## Define a 5-Node Chain

We'll use a chain **A → B → C → D → E**. Node A has a strong prior;
the rest are weak (uninformative). Four implications propagate belief
along the chain.

With `max_block_size=3`, this graph is too large for a single block,
so the partitioner must split it.

In [ ]:
priors = {
    "A": {"strength": 0.8, "confidence": 0.9},
    "B": {"strength": 0.5, "confidence": 0.01},
    "C": {"strength": 0.5, "confidence": 0.01},
    "D": {"strength": 0.5, "confidence": 0.01},
    "E": {"strength": 0.5, "confidence": 0.01},
}
implications = [
    {"src": "A", "dst": "B", "strength": 0.9, "confidence": 0.9},
    {"src": "B", "dst": "C", "strength": 0.8, "confidence": 0.85},
    {"src": "C", "dst": "D", "strength": 0.8, "confidence": 0.85},
    {"src": "D", "dst": "E", "strength": 0.8, "confidence": 0.85},
]

---
## Partition the Graph

The partitioner sorts edges by confidence (ascending) and removes
the weakest-confidence edge first until every connected component
fits within `max_block_size`. Boundary nodes — endpoints of cut
edges — will receive inter-block messages.

In [ ]:
partition = partition_into_blocks(priors, implications, max_block_size=3)

print(f"Blocks ({len(partition.blocks)}):")
for i, block in enumerate(partition.blocks):
    print(f"  Block {i}: {block}")
print(f"\nBoundary nodes: {dict(partition.boundary_nodes)}")
print(f"Cut edges: {len(partition.cut_edges)}")
print(f"Has cycle: {partition.has_cycle}")

The partitioner cut the weakest-confidence edge(s) first.
Because this is a tree-structured chain (no cycles), convergence
is guaranteed — Pearl's belief propagation converges exactly on trees.

---
## Run Block-Diagonal Inference

Each block is sampled independently via Gibbs, then boundary marginals
are exchanged as soft-clamp factors. The loop repeats until KL
divergence between successive messages drops below threshold.

In [ ]:
result = run_block_diagonal_sampling(
    priors, implications, k=4, max_block_size=3, seed=42,
)

print(f"Converged: {result.converged} (in {result.n_iterations} iterations)")
print()
print(f"{'Node':<6} {'Strength':>10} {'Confidence':>12}")
print(f"{'\u2500' * 28}")
for name in ["A", "B", "C", "D", "E"]:
    print(f"{name:<6} {result.strengths[name]:>10.4f} {result.confidences[name]:>12.4f}")

Notice how **strength attenuates** along the chain — each implication
has strength < 1.0, so the belief weakens with distance from A.
**Confidence also drops**: less evidence reaches distant nodes.

---
## Compare with Full-Graph Baseline

To validate the block-diagonal approximation, we compare against
full-graph inference — all 5 nodes in a single factor graph with
the same K=4 discretization.

In [ ]:
full_graph = build_beta_full_graph(priors, implications, k=4)
full_samples = run_beta_sampling(full_graph, seed=42)

print(f"{'Node':<6} {'Full-graph s':>14} {'Block-diag s':>14} {'\u0394s':>8}")
print(f"{'\u2500' * 42}")
for name in ["B", "C", "D", "E"]:
    _, s_full, _ = estimate_beta_marginal(
        full_samples, full_graph, full_graph["nodes"][name])
    s_bd = result.strengths[name]
    print(f"{name:<6} {s_full:>14.4f} {s_bd:>14.4f} {abs(s_full - s_bd):>8.4f}")

Block-diagonal matches full-graph well. The small differences come
from discretizing inter-block messages into K=4 bins — information
is lossily compressed at each block boundary.

---
## Convenience Wrapper

For a single target node, `sample_and_measure_block_diagonal`
wraps the full pipeline into one call.

In [ ]:
# For a single target node, use the convenience wrapper:
s, c = sample_and_measure_block_diagonal(
    priors, implications, "E", k=4, max_block_size=3, seed=42)
print(f"P(E) = (stv {s:.4f} {c:.4f})")

---
## Next Steps

- **Diamond graph (cyclic)**: the partitioner detects cycles and uses damped loopy BP
- **K=4** keeps couplings to 16 per implication — fits within TSU's 12-connection budget
- See `tests/test_block_diagonal.py` for more topologies including diamond graphs